In [6]:
%reset -f

In [11]:
from src.optimization.core import (
    load_demands,
    build_milp,
    solve_model,
    save_optimization_results,
    extract_solver_solution
)
from src.sampling.core import load_samples
from src.misc.constants import PROJ_ROOT
import pyomo.environ as pyo

In [8]:
csv_file = PROJ_ROOT / "energy_demands.csv"
Q_D, P_D = load_demands(csv_file)

# Load the training samples
lhs_df, _ = load_samples("lhs", "training")
sobol_df, _ = load_samples("sobol", "training")

sample_sets = {
    "LHS": lhs_df,
    # "Sobol": sobol_df
}


In [9]:
c_g, c_el = sample_sets["LHS"]["gas_price"].iloc[0] / 1000, sample_sets["LHS"]["electricity_price"].iloc[0] / 1000

model = build_milp(Q_D, P_D, c_g, c_el)

results = solve_model(model, MIPGap=1e-3, TimeLimit=120)

Read LP format model from file C:\Users\erdem\AppData\Local\Temp\tmpjz__0zbd.pyomo.lp
Reading time = 0.02 seconds
x1: 4371 rows, 3697 columns, 10756 nonzeros
Set parameter MIPGap to value 0.001
Set parameter TimeLimit to value 120
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: AMD Ryzen 3 7330U with Radeon Graphics, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  120
MIPGap  0.001

Optimize a model with 4371 rows, 3697 columns and 10756 nonzeros (Min)
Model fingerprint: 0x348f4306
Model has 336 linear objective coefficients
Variable types: 2689 continuous, 1008 integer (1008 binary)
Coefficient statistics:
  Matrix range     [4e-01, 1e+03]
  Objective range  [3e-02, 7e-02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+03]

Presolve removed 2337 rows and 1681 columns
Presolve time: 0.03s
Presolved: 2034 rows, 2016 columns, 5866

In [12]:
sol_df = extract_solver_solution(model)